# GeoLifeCLEF 2025 — reproducible research pipeline

Single control notebook for matched baselines, Landsat experiments, spatial validation, multi-seed ablations, climate models and fusion. Every stage writes restartable artifacts and uses only competition data.

In [ ]:
from pathlib import Path
import json, subprocess, sys

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
DATA_ROOT = Path('/kaggle/input/competitions/geolifeclef-2025')
assert DATA_ROOT.is_dir(), 'Attach the geolifeclef-2025 competition source.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', str(ROOT)], check=True)

In [ ]:
def run_once(output, command):
    output = ROOT / output
    if output.exists():
        print(f'Reusing {output}')
        return
    subprocess.run(command, cwd=ROOT, check=True)

def read_json(path):
    return json.loads((ROOT / path).read_text(encoding='utf-8'))

## 1. Full frequency baseline

In [ ]:
run_once('artifacts/frequency_pa/metrics.json', [sys.executable, 'scripts/run_frequency_baseline.py', '--metadata-path', str(DATA_ROOT / 'GLC25_PA_metadata_train.csv'), '--report-path', 'artifacts/frequency_pa/metrics.json'])
read_json('artifacts/frequency_pa/metrics.json')

## 2. Full Landsat temporal CNN

In [ ]:
run_once('data/processed/landsat_pa_full_manifest.json', [sys.executable, 'scripts/prepare_landsat_pa.py', '--data-root', str(DATA_ROOT), '--train-output', 'data/processed/landsat_pa_full_train.npz', '--val-output', 'data/processed/landsat_pa_full_val.npz', '--manifest-path', 'data/processed/landsat_pa_full_manifest.json', '--max-train-surveys', '71190', '--max-validation-surveys', '17797'])
run_once('artifacts/landsat_tcn_full/metrics.json', [sys.executable, 'scripts/train.py', '--config', 'configs/landsat_tcn_full.yaml'])
run_once('artifacts/landsat_tcn_full/evaluation.json', [sys.executable, 'scripts/evaluate.py', '--checkpoint', 'artifacts/landsat_tcn_full/best.pt', '--split', 'data/processed/landsat_pa_full_val.npz', '--channels', '32', '--top-k', '16'])

In [ ]:
baseline = read_json('artifacts/frequency_pa/metrics.json')
landsat = read_json('artifacts/landsat_tcn_full/evaluation.json')
manifest = read_json('data/processed/landsat_pa_full_manifest.json')
summary = {'train_surveys': manifest['train_samples'], 'validation_surveys': manifest['validation_samples'], 'frequency_sample_f1': baseline.get('sample_f1_top_k'), 'landsat_top16_sample_f1': landsat.get('sample_f1_top_k'), 'official_private_leaderboard_target': 0.2302}
summary

## 3. Spatial holdout audit

Choose the geographic holdout with a pre-registered rule, before inspecting model performance. This prevents selecting an easy region after the fact.

In [ ]:
run_once('data/reports/spatial_split_audit.json', [sys.executable, 'scripts/audit_spatial_split.py', '--metadata-path', str(DATA_ROOT / 'GLC25_PA_metadata_train.csv'), '--report-path', 'data/reports/spatial_split_audit.json'])
spatial_audit = read_json('data/reports/spatial_split_audit.json')
{'countries': spatial_audit['country_count'], 'regions': spatial_audit['region_count'], 'recommended_holdout': spatial_audit['recommended_holdout']}

## 4. Next registered stages

After the spatial split is frozen, this same notebook will run the matched spatial frequency/Landsat comparison, multi-seed ablations, climate-only model, Landsat + climate gated fusion, and calibration-only top-k selection.